# Comparing embodiments with `BaseEnvConfig.sweep`

One task, one seed, arms with 2 to 5 joints that share the same total length and mass. `sweep` expands a base config into a family, one `EnvConfig`
(and one `env_hash`) per embodiment; launching jobs is up to you. Every item here is rebuilt from its resolved dict, the way a worker would.

The task is the package's `empty/idle` placeholder, so the example needs only the `env` extra (no robosuite): what differs between the rows is the
embodiment.

In [1]:
import numpy as np
import torch

from robotdesigner.config import BaseEnvConfig, Chain, EnvConfig, FixedGains
from robotdesigner.envs import make_env

base = (
    BaseEnvConfig()
    .robot(Chain(n=3, axes="zyy", total_length=0.9, total_mass=3.0))   # the same total length and mass for every n
    .control(gains=FixedGains(kp=60.0, kd=6.0))
    .task(horizon=40)
    .sim(num_envs=4, scene="plain_floor")
)
family = base.sweep(
    paired={"robot.n": [2, 3, 4, 5], "robot.axes": ["zy", "zyy", "zyyy", "zyyyy"]},   # zipped: four embodiments
    grid={"sim.seed": range(1)},                                                    # one seed; add range(3) for three
)
print(family, "| notes:", family.notes)

Family(4 configs) | notes: []


## The config table

`table()` gives the override values and the `env_hash` of every item. `n` and `axes` are *paired*, so they change together; a grid over them would
also try `n=2` with three axis letters, which `sweep` rejects at expansion with the item index and the override.

In [2]:
for row in family.table():
    print(row)

{'index': 0, 'robot.n': 2, 'robot.axes': 'zy', 'sim.seed': 0, 'env_hash': 'e2aa3648c089'}
{'index': 1, 'robot.n': 3, 'robot.axes': 'zyy', 'sim.seed': 0, 'env_hash': '0368b76d75ae'}
{'index': 2, 'robot.n': 4, 'robot.axes': 'zyyy', 'sim.seed': 0, 'env_hash': '9c7719e775ff'}
{'index': 3, 'robot.n': 5, 'robot.axes': 'zyyyy', 'sim.seed': 0, 'env_hash': 'cd1b7294cbe8'}


## A short rollout of each

`configs()` are the resolved `EnvConfig`s. A worker would receive `resolve().to_dict()` and rebuild with `EnvConfig.from_dict`, as done here. The
same seed and the same random actions drive every embodiment, so the rows differ by the arm only.

In [3]:
print(f"{'n':>2}  {'env_hash':<12}  {'robot_hash':<12}  {'action dim':>10}  {'mean |q| after 40 steps (rad)':>30}")
for cfg in family.configs():
    cfg = EnvConfig.from_dict(cfg.to_dict())          # what a worker does
    env = make_env(cfg)
    env.set_seed(0)
    torch.manual_seed(0)
    traj = env.rollout(40)
    q = traj["next", "proprio", "qpos"][:, -1]
    print(f"{cfg.robot.n:>2}  {cfg.env_hash:<12}  {env.robot_hash:<12}  {env.action_spec.shape[-1]:>10}  {q.abs().mean().item():>30.3f}")
    env.close()

 n  env_hash      robot_hash    action dim   mean |q| after 40 steps (rad)


 2  e2aa3648c089  0fbaa4867266           2                           0.088
 3  0368b76d75ae  287b31f43ec4           3                           0.378
 4  9c7719e775ff  723e793ab2fd           4                           0.297
 5  cd1b7294cbe8  b1e089998f9b           5                           0.269


## Other axes

Any dotted path works, including the simulator's options and the task's params. A grid multiplies with the paired group; the order is deterministic
(the paired group first, then the grid axes in the order given, the last one fastest), and items that resolve to the same `env_hash` are dropped with a note.

In [4]:
more = base.sweep(
    paired={"robot.n": [2, 3], "robot.axes": ["zy", "zyy"]},
    grid={"sim.seed": range(2), "sim.options.self_collision": [False, True]},
)
for row in more.table():
    print(row)

{'index': 0, 'robot.n': 2, 'robot.axes': 'zy', 'sim.seed': 0, 'sim.options.self_collision': False, 'env_hash': 'e2aa3648c089'}
{'index': 1, 'robot.n': 2, 'robot.axes': 'zy', 'sim.seed': 0, 'sim.options.self_collision': True, 'env_hash': '2658530a4fde'}
{'index': 2, 'robot.n': 2, 'robot.axes': 'zy', 'sim.seed': 1, 'sim.options.self_collision': False, 'env_hash': 'b756ef0adc8c'}
{'index': 3, 'robot.n': 2, 'robot.axes': 'zy', 'sim.seed': 1, 'sim.options.self_collision': True, 'env_hash': 'ebb822278c94'}
{'index': 4, 'robot.n': 3, 'robot.axes': 'zyy', 'sim.seed': 0, 'sim.options.self_collision': False, 'env_hash': '0368b76d75ae'}
{'index': 5, 'robot.n': 3, 'robot.axes': 'zyy', 'sim.seed': 0, 'sim.options.self_collision': True, 'env_hash': 'b0aa32870b8a'}
{'index': 6, 'robot.n': 3, 'robot.axes': 'zyy', 'sim.seed': 1, 'sim.options.self_collision': False, 'env_hash': 'e17748071f14'}
{'index': 7, 'robot.n': 3, 'robot.axes': 'zyy', 'sim.seed': 1, 'sim.options.self_collision': True, 'env_hash': 